# Notebook #6 — Session Manipulation Reaction
## استراتژی واکنش به Manipulation در سشن‌های معاملاتی — XAUUSD (M5)

---

### فلسفه Session Manipulation
در مارکت XAUUSD، الگوی مشخصی وجود دارد:

**ساعت‌های اول هر سشن** = منطقه Manipulation:
- بازار ابتدا به سمت اشتباه حرکت می‌کند (False Move)
- SL‌ها زده می‌شوند
- سپس حرکت اصلی (Expansion) آغاز می‌شود

### سه سشن:
| سشن | ساعت UTC | ویژگی |
|---|---|---|
| Tokyo/Asia | 00:00-08:00 | کم‌حجم، range محدود |
| London | 07:00-16:00 | اول کار: Manipulation |
| New York | 13:00-22:00 | Expansion اصلی |

### الگوی کلی:
```
[Asia Range] → شناسایی High/Low آسیا
→ [London Open] → قیمت High یا Low آسیا را می‌زند (Manipulation)
→ [Reversal] → برگشت قوی (Expansion)
→ Entry بعد از تأیید
```

### انواع Manipulation:
1. **London Sweep of Asia High** → بعد از sweep: BUY
2. **London Sweep of Asia Low** → بعد از sweep: SELL
3. **NY Sweep of London Range** → ورود در جهت Expansion

| پارامتر | مقدار |
|---|---|
| نماد | XAUUSD |
| تایم‌فریم | M5 |
| Asia Session | 00:00-07:00 UTC |
| London Open | 07:00-10:00 UTC |
| NY Open | 13:00-16:00 UTC |
| Risk/Reward | 1:2 |

## Step 1 — Imports & Configuration

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, Tuple

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook'
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

SYMBOL        = 'XAUUSD'
DATA_DIR      = Path('./data')
LOOKBACK_DAYS = 30

# ── Session Times (UTC — broker time stored in CSV) ─────────────────────────
# Adjust if broker uses different timezone offset
ASIA_START_H    =  0
ASIA_END_H      =  7
LONDON_START_H  =  7
LONDON_OPEN_H   = 10   # London manipulation window: 07-10
NY_START_H      = 13
NY_OPEN_H       = 16   # NY manipulation window: 13-16

# ── Strategy Parameters ──────────────────────────────────────────────────────
SWEEP_MIN        = 0.5    # minimum sweep beyond Asia range (USD)
RETURN_MAX_BARS  = 12     # max M5 bars for price to return inside range
CONFIRM_BARS     = 2      # M5 bars closing in correct direction
SWING_WINDOW     = 5

# ── Risk ─────────────────────────────────────────────────────────────────────
RISK_REWARD    = 2.0
SL_BUFFER      = 0.5
MAX_TRADE_BARS = 144

print('Session Manipulation Strategy — Config loaded.')
print(f'  Asia   : {ASIA_START_H:02d}:00-{ASIA_END_H:02d}:00 UTC')
print(f'  London : {LONDON_START_H:02d}:00 UTC (manipulation: {LONDON_START_H:02d}:00-{LONDON_OPEN_H:02d}:00)')
print(f'  NY     : {NY_START_H:02d}:00 UTC (manipulation: {NY_START_H:02d}:00-{NY_OPEN_H:02d}:00)')

Session Manipulation Strategy — Config loaded.
  Asia   : 00:00-07:00 UTC
  London : 07:00 UTC (manipulation: 07:00-10:00)
  NY     : 13:00 UTC (manipulation: 13:00-16:00)


## Step 2 — Load Data

In [2]:
def load_ohlcv(symbol: str, tf: str, lookback_days: int) -> pd.DataFrame:
    path = DATA_DIR / symbol / tf / 'ohlcv.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}')
    df = pd.read_csv(path)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df = df.sort_values('time').reset_index(drop=True)
    keep = ['time', 'open', 'high', 'low', 'close', 'tick_volume']
    df = df[[c for c in keep if c in df.columns]].copy()
    df.rename(columns={'tick_volume': 'volume'}, inplace=True)
    cutoff = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=lookback_days)
    return df[df['time'] >= cutoff].copy().reset_index(drop=True)


df_m5 = load_ohlcv(SYMBOL, 'M5', LOOKBACK_DAYS)

# Add time columns
df_m5['date']    = df_m5['time'].dt.date
df_m5['hour']    = df_m5['time'].dt.hour
df_m5['minute']  = df_m5['time'].dt.minute
df_m5['weekday'] = df_m5['time'].dt.weekday
df_m5['is_weekday'] = df_m5['weekday'] < 5

print(f'M5 bars: {len(df_m5):,}')
trading_days = df_m5[df_m5['is_weekday']]['date'].unique()
print(f'Trading days: {len(trading_days)}')

M5 bars: 5,888
Trading days: 22


## Step 3 — Asia Session Range Extraction

برای هر روز معاملاتی:
- High و Low کندل‌های Asia (00:00-07:00 UTC) = **Asia Range**
- این Range = محل نقدینگی برای سشن London

In [3]:
def build_asia_ranges(df_m5: pd.DataFrame) -> pd.DataFrame:
    """Build daily Asia session range (00:00-07:00 UTC)."""
    records = []
    for day in sorted(df_m5[df_m5['is_weekday']]['date'].unique()):
        asia_bars = df_m5[
            (df_m5['date'] == day) &
            (df_m5['hour'] >= ASIA_START_H) &
            (df_m5['hour'] < ASIA_END_H)
        ]
        if asia_bars.empty or len(asia_bars) < 12:  # at least 1 hour of data
            continue

        records.append({
            'date'        : day,
            'asia_high'   : round(asia_bars['high'].max(), 2),
            'asia_low'    : round(asia_bars['low'].min(), 2),
            'asia_range'  : round(asia_bars['high'].max() - asia_bars['low'].min(), 2),
            'asia_open'   : round(asia_bars.iloc[0]['open'], 2),
            'asia_close'  : round(asia_bars.iloc[-1]['close'], 2),
            'asia_bars'   : len(asia_bars),
            'london_start': pd.Timestamp(str(day) + f' {LONDON_START_H:02d}:00:00', tz='UTC'),
            'ny_start'    : pd.Timestamp(str(day) + f' {NY_START_H:02d}:00:00', tz='UTC'),
        })

    asia_df = pd.DataFrame(records)
    print(f'Asia ranges built: {len(asia_df)}')
    print(f'  Avg range : {asia_df["asia_range"].mean():.2f}')
    print(f'  Min range : {asia_df["asia_range"].min():.2f}')
    print(f'  Max range : {asia_df["asia_range"].max():.2f}')
    return asia_df


asia_ranges = build_asia_ranges(df_m5)
display(asia_ranges.head(5))

Asia ranges built: 21
  Avg range : 47.00
  Min range : 21.64
  Max range : 105.59


,date,asia_high,asia_low,asia_range,asia_open,asia_close,asia_bars,london_start,ny_start
0,2026-04-17,4801.2300,4767.7700,33.4600,4791.4700,4793.1800,72,2026-04-17 07:00:00+00:00,2026-04-17 13:00:00+00:00
1,2026-04-20,4814.3600,4737.0500,77.3100,4781.5700,4790.9000,71,2026-04-20 07:00:00+00:00,2026-04-20 13:00:00+00:00
2,2026-04-21,4833.1000,4792.6500,40.4500,4822.1800,4797.1100,72,2026-04-21 07:00:00+00:00,2026-04-21 13:00:00+00:00
3,2026-04-22,4765.0500,4715.6200,49.4300,4720.5700,4758.2400,72,2026-04-22 07:00:00+00:00,2026-04-22 13:00:00+00:00
4,2026-04-23,4753.4400,4694.2800,59.1600,4738.3700,4703.9900,72,2026-04-23 07:00:00+00:00,2026-04-23 13:00:00+00:00


## Step 4 — Manipulation Detection

### London Sweep Pattern:
```
Asia High ─────────────────────────────
           ↑ Sweep (London takes it)    
           │ Price returns below Asia High
           ↓ Expansion Move DOWN
Asia Low  ─────────────────────────────
```

بعد از Sweep بالای Asia High → Sell
بعد از Sweep پایین Asia Low → Buy

In [4]:
def detect_session_manipulation(
    df_m5: pd.DataFrame,
    asia_ranges: pd.DataFrame,
) -> list:
    """
    For each day, check if London/NY session manipulates the Asia range.

    Flow:
    1. London opens: monitor for sweep of Asia High or Low
    2. Sweep: price briefly goes beyond Asia range
    3. Return: price closes back inside (or crosses in reverse)
    4. Expansion: continue in reverse direction
    5. Entry: at close of return candle
    """
    manipulation_events = []

    for _, asia in asia_ranges.iterrows():
        day       = asia['date']
        ah        = asia['asia_high']
        al        = asia['asia_low']
        ar        = asia['asia_range']
        lon_start = asia['london_start']
        ny_start  = asia['ny_start']

        lon_bars = df_m5[
            (df_m5['date'] == day) &
            (df_m5['hour'] >= LONDON_START_H) &
            (df_m5['hour'] < LONDON_OPEN_H)
        ].reset_index(drop=True)

        ny_bars = df_m5[
            (df_m5['date'] == day) &
            (df_m5['hour'] >= NY_START_H) &
            (df_m5['hour'] < NY_OPEN_H)
        ].reset_index(drop=True)

        for session_label, session_bars in [('LONDON', lon_bars), ('NY', ny_bars)]:
            if session_bars.empty:
                continue

            swept_high = False
            swept_low  = False
            sweep_idx  = None

            for i, bar in session_bars.iterrows():
                if (not swept_high and not swept_low and
                        bar['high'] > ah + SWEEP_MIN and ar > 1.0):
                    swept_high = True
                    sweep_idx  = i

                elif swept_high:
                    bars_since = i - sweep_idx
                    if bars_since > RETURN_MAX_BARS:
                        swept_high = False; sweep_idx = None; continue
                    if bar['close'] < ah:
                        sweep_candle = session_bars.iloc[sweep_idx]
                        manipulation_events.append({
                            'date'         : day,
                            'session'      : session_label,
                            'manip_type'   : 'high_sweep',
                            'direction'    : 'SELL',
                            'asia_high'    : ah,
                            'asia_low'     : al,
                            'asia_range'   : ar,
                            'sweep_time'   : sweep_candle['time'],
                            'sweep_high'   : sweep_candle['high'],
                            'entry_idx'    : df_m5[df_m5['time'] == bar['time']].index[0]
                                            if len(df_m5[df_m5['time'] == bar['time']]) > 0 else None,
                            'entry_time'   : bar['time'],
                            'entry_price'  : round(bar['close'], 2),
                            'swept_level'  : ah,
                        })
                        swept_high = False; sweep_idx = None
                        break

                elif (not swept_high and not swept_low and
                          bar['low'] < al - SWEEP_MIN and ar > 1.0):
                    swept_low = True
                    sweep_idx = i

                elif swept_low:
                    bars_since = i - sweep_idx
                    if bars_since > RETURN_MAX_BARS:
                        swept_low = False; sweep_idx = None; continue
                    if bar['close'] > al:
                        sweep_candle = session_bars.iloc[sweep_idx]
                        manipulation_events.append({
                            'date'         : day,
                            'session'      : session_label,
                            'manip_type'   : 'low_sweep',
                            'direction'    : 'BUY',
                            'asia_high'    : ah,
                            'asia_low'     : al,
                            'asia_range'   : ar,
                            'sweep_time'   : sweep_candle['time'],
                            'sweep_low'    : sweep_candle['low'],
                            'entry_idx'    : df_m5[df_m5['time'] == bar['time']].index[0]
                                            if len(df_m5[df_m5['time'] == bar['time']]) > 0 else None,
                            'entry_time'   : bar['time'],
                            'entry_price'  : round(bar['close'], 2),
                            'swept_level'  : al,
                        })
                        swept_low = False; sweep_idx = None
                        break

    return manipulation_events


events = detect_session_manipulation(df_m5, asia_ranges)

print(f'Manipulation events: {len(events)}')
high_sweeps = [e for e in events if e['manip_type'] == 'high_sweep']
low_sweeps  = [e for e in events if e['manip_type'] == 'low_sweep']
print(f'  High sweeps (→SELL): {len(high_sweeps)}')
print(f'  Low sweeps  (→BUY) : {len(low_sweeps)}')
lon_events  = [e for e in events if e['session'] == 'LONDON']
ny_events   = [e for e in events if e['session'] == 'NY']
print(f'  London events: {len(lon_events)}')
print(f'  NY events    : {len(ny_events)}')

Manipulation events: 23
  High sweeps (→SELL): 11
  Low sweeps  (→BUY) : 12
  London events: 15
  NY events    : 8


## Step 5 — Backtesting Engine

In [5]:
def simulate_trade(
    df: pd.DataFrame,
    entry_idx: int,
    direction: str,
    entry: float,
    sl: float,
    tp: float,
) -> dict:
    bars = df.iloc[entry_idx + 1 : entry_idx + MAX_TRADE_BARS + 1]
    for i, bar in enumerate(bars.itertuples(), 1):
        if direction == 'BUY':
            if bar.low <= sl:
                return {'result': 'SL', 'exit_price': sl, 'exit_time': bar.time, 'pnl_r': -1.0, 'bars_held': i}
            if bar.high >= tp:
                return {'result': 'TP', 'exit_price': tp, 'exit_time': bar.time, 'pnl_r': RISK_REWARD, 'bars_held': i}
        else:
            if bar.high >= sl:
                return {'result': 'SL', 'exit_price': sl, 'exit_time': bar.time, 'pnl_r': -1.0, 'bars_held': i}
            if bar.low <= tp:
                return {'result': 'TP', 'exit_price': tp, 'exit_time': bar.time, 'pnl_r': RISK_REWARD, 'bars_held': i}
    if not bars.empty:
        last = bars.iloc[-1]
        risk = abs(entry - sl)
        pnl  = ((last['close'] - entry) / risk if direction == 'BUY'
                else (entry - last['close']) / risk)
        return {'result': 'OPEN', 'exit_price': round(last['close'], 2),
                'exit_time': last['time'], 'pnl_r': round(pnl, 3), 'bars_held': len(bars)}
    return {'result': 'OPEN', 'exit_price': entry,
            'exit_time': df.iloc[entry_idx]['time'], 'pnl_r': 0.0, 'bars_held': 0}


def run_session_backtest(df_m5: pd.DataFrame, events: list) -> pd.DataFrame:
    """
    For each manipulation event:
    - SL: beyond the sweep extremity
    - TP: 2R
    - Entry: at event's entry_price
    """
    trades = []

    for event in events:
        if event['entry_idx'] is None:
            continue

        entry_idx = event['entry_idx']
        if entry_idx >= len(df_m5) - 1:
            continue

        direction = event['direction']
        entry     = event['entry_price']

        if direction == 'SELL':  # high sweep → sell
            swept_h   = event.get('sweep_high', event['swept_level'] + SWEEP_MIN)
            sl        = max(swept_h, event['asia_high']) + SL_BUFFER
            risk      = sl - entry
        else:  # low sweep → buy
            swept_l   = event.get('sweep_low', event['swept_level'] - SWEEP_MIN)
            sl        = min(swept_l, event['asia_low']) - SL_BUFFER
            risk      = entry - sl

        if risk <= 0:
            continue

        tp = (entry + RISK_REWARD * risk if direction == 'BUY'
              else entry - RISK_REWARD * risk)

        outcome = simulate_trade(df_m5, entry_idx, direction, entry, sl, tp)

        trades.append({
            'date'        : event['date'],
            'session'     : event['session'],
            'manip_type'  : event['manip_type'],
            'direction'   : direction,
            'asia_high'   : event['asia_high'],
            'asia_low'    : event['asia_low'],
            'asia_range'  : event['asia_range'],
            'sweep_time'  : event['sweep_time'],
            'entry_time'  : event['entry_time'],
            'entry_price' : entry,
            'sl'          : round(sl, 2),
            'tp'          : round(tp, 2),
            'risk'        : round(risk, 2),
            **outcome,
        })

    return pd.DataFrame(trades) if trades else pd.DataFrame()


trades_df = run_session_backtest(df_m5, events)

if trades_df.empty:
    print('No trades generated.')
else:
    print(f'Total trades: {len(trades_df)}')
    for session in ['LONDON', 'NY']:
        sub = trades_df[trades_df['session'] == session]
        if sub.empty: continue
        closed = sub[sub['result'].isin(['TP','SL'])]
        wr = (closed['result']=='TP').mean() * 100 if len(closed) > 0 else 0
        print(f'  {session}: n={len(sub)}  WR={wr:.0f}%')
    display(trades_df.head(5))

Total trades: 23
  LONDON: n=15  WR=33%
  NY: n=8  WR=38%


,date,session,manip_type,direction,asia_high,asia_low,asia_range,sweep_time,entry_time,entry_price,sl,tp,risk,result,exit_price,exit_time,pnl_r,bars_held
0,2026-04-17,LONDON,high_sweep,SELL,4801.2300,4767.7700,33.4600,2026-04-17 07:55:00+00:00,2026-04-17 08:00:00+00:00,4801.1600,4805.0500,4793.3800,3.8900,SL,4805.0500,2026-04-17 08:05:00+00:00,-1.0000,1
1,2026-04-17,NY,high_sweep,SELL,4801.2300,4767.7700,33.4600,2026-04-17 14:45:00+00:00,2026-04-17 14:55:00+00:00,4799.2200,4812.1900,4773.2800,12.9700,SL,4812.1900,2026-04-17 15:20:00+00:00,-1.0000,5
2,2026-04-20,NY,high_sweep,SELL,4814.3600,4737.0500,77.3100,2026-04-20 15:15:00+00:00,2026-04-20 15:20:00+00:00,4813.8700,4816.2700,4809.0700,2.4000,TP,4809.0700,2026-04-20 15:25:00+00:00,2.0000,1
3,2026-04-21,LONDON,low_sweep,BUY,4833.1000,4792.6500,40.4500,2026-04-21 07:20:00+00:00,2026-04-21 07:35:00+00:00,4792.9900,4790.8600,4797.2500,2.1300,SL,4790.8600,2026-04-21 07:45:00+00:00,-1.0000,2
4,2026-04-21,NY,low_sweep,BUY,4833.1000,4792.6500,40.4500,2026-04-21 14:10:00+00:00,2026-04-21 14:40:00+00:00,4793.1200,4778.7300,4821.9000,14.3900,SL,4778.7300,2026-04-21 15:30:00+00:00,-1.0000,10


## Step 6 — Performance Analytics

In [6]:
def calc_metrics(trades_df: pd.DataFrame) -> dict:
    if trades_df.empty:
        return {}
    closed = trades_df[trades_df['result'].isin(['TP', 'SL'])].copy()
    if closed.empty:
        return {}
    n   = len(closed)
    wins = (closed['result'] == 'TP').sum()
    wr  = wins / n
    closed['cum_r'] = closed['pnl_r'].cumsum()
    dd  = closed['cum_r'] - closed['cum_r'].cummax()
    pos = closed[closed['pnl_r'] > 0]['pnl_r'].sum()
    neg = abs(closed[closed['pnl_r'] < 0]['pnl_r'].sum())
    pf  = pos / neg if neg > 0 else float('inf')
    return {
        'total_trades'   : n,
        'wins'           : int(wins),
        'losses'         : n - int(wins),
        'win_rate'       : wr,
        'total_r'        : round(closed['pnl_r'].sum(), 3),
        'avg_r'          : round(closed['pnl_r'].mean(), 3),
        'profit_factor'  : round(pf, 3),
        'max_dd_r'       : round(dd.min(), 3),
        'expectancy'     : round(wr * RISK_REWARD - (1 - wr), 3),
        'avg_bars'       : round(closed['bars_held'].mean(), 1),
        'cum_r'          : closed['cum_r'].reset_index(drop=True),
        'drawdown'       : dd.reset_index(drop=True),
        'closed'         : closed,
    }


metrics = calc_metrics(trades_df)

if metrics and metrics.get('total_trades', 0) > 0:
    sep = '=' * 55
    print(sep)
    print('  PERFORMANCE — Session Manipulation Reaction')
    print(sep)
    print(f'  Trades        : {metrics["total_trades"]}')
    print(f'  Wins / Losses : {metrics["wins"]} / {metrics["losses"]}')
    print(f'  Win Rate      : {metrics["win_rate"]*100:.1f}%')
    print(f'  Total R       : {metrics["total_r"]:+.2f} R')
    print(f'  Profit Factor : {metrics["profit_factor"]:.2f}')
    print(f'  Expectancy    : {metrics["expectancy"]:+.3f} R')
    print(f'  Max Drawdown  : {metrics["max_dd_r"]:.2f} R')
    print(f'  Avg Duration  : {metrics["avg_bars"]:.0f} M5 bars')
    print()
    print('  By Session:')
    for sess in ['LONDON', 'NY']:
        sub = metrics['closed'][metrics['closed']['session'] == sess]
        if sub.empty: continue
        wr_s = (sub['result'] == 'TP').mean()
        print(f'    {sess:8s}: n={len(sub):3d}  WR={wr_s*100:.0f}%  AvgR={sub["pnl_r"].mean():+.3f}')
    print('  By Manipulation Type:')
    for mt in ['high_sweep', 'low_sweep']:
        sub = metrics['closed'][metrics['closed']['manip_type'] == mt]
        if sub.empty: continue
        wr_m = (sub['result'] == 'TP').mean()
        print(f'    {mt:12s}: n={len(sub):3d}  WR={wr_m*100:.0f}%  AvgR={sub["pnl_r"].mean():+.3f}')
    print(sep)

  PERFORMANCE — Session Manipulation Reaction
  Trades        : 23
  Wins / Losses : 8 / 15
  Win Rate      : 34.8%
  Total R       : +1.00 R
  Profit Factor : 1.07
  Expectancy    : +0.043 R
  Max Drawdown  : -5.00 R
  Avg Duration  : 8 M5 bars

  By Session:
    LONDON  : n= 15  WR=33%  AvgR=+0.000
    NY      : n=  8  WR=38%  AvgR=+0.125
  By Manipulation Type:
    high_sweep  : n= 11  WR=27%  AvgR=-0.182
    low_sweep   : n= 12  WR=42%  AvgR=+0.250


## Step 7 — Visualizations

### 7.1 — Asia Range + Manipulation Chart

In [7]:
def plot_session_day(day_date, df_m5: pd.DataFrame, asia_ranges: pd.DataFrame,
                     trades_df: pd.DataFrame) -> None:
    """Full day M5 chart with Asia range and session markers."""
    asia = asia_ranges[asia_ranges['date'] == day_date]
    if asia.empty:
        print(f'No Asia range for {day_date}')
        return
    asia = asia.iloc[0]

    day_m5 = df_m5[df_m5['date'] == day_date].copy()
    if day_m5.empty:
        return

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=day_m5['time'], open=day_m5['open'], high=day_m5['high'],
        low=day_m5['low'], close=day_m5['close'],
        name='M5',
        increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350',
    ))

    # Asia range band
    fig.add_hrect(
        y0=asia['asia_low'], y1=asia['asia_high'],
        fillcolor='rgba(255,152,0,0.08)',
        line=dict(color='rgba(255,152,0,0.4)', width=1),
        annotation_text='Asia Range',
        annotation_position='top left',
    )
    fig.add_hline(y=asia['asia_high'], line_color='rgba(255,152,0,0.8)',
                  line_dash='dash', line_width=2,
                  annotation_text=f'Asia High {asia["asia_high"]:.2f}')
    fig.add_hline(y=asia['asia_low'], line_color='rgba(33,150,243,0.8)',
                  line_dash='dash', line_width=2,
                  annotation_text=f'Asia Low {asia["asia_low"]:.2f}')

    # Session markers
    t_str = str(day_date)
    for sess_h, sess_name, color in [
        (LONDON_START_H, 'London Open', 'rgba(0,230,118,0.8)'),
        (NY_START_H,     'NY Open',     'rgba(0,176,255,0.8)'),
    ]:
        sess_t = pd.Timestamp(f'{t_str} {sess_h:02d}:00:00', tz='UTC')
        if sess_t <= day_m5['time'].max():
            fig.add_vline(x=sess_t, line_color=color, line_dash='dot',
                          annotation_text=sess_name, annotation_position='top')

    # Trade entries for this day
    if not trades_df.empty:
        day_trades = trades_df[trades_df['date'] == day_date]
        for _, t in day_trades.iterrows():
            ec   = '#26a69a' if t['direction'] == 'BUY' else '#ef5350'
            rc   = '#00E676' if t['result'] == 'TP' else '#FF1744'
            esym = 'triangle-up' if t['direction'] == 'BUY' else 'triangle-down'
            fig.add_trace(go.Scatter(
                x=[t['entry_time']], y=[t['entry_price']],
                mode='markers+text', showlegend=False,
                marker=dict(symbol=esym, size=16, color=rc,
                            line=dict(color='white', width=2)),
                text=[f'{t["direction"]}\n{t["entry_price"]:.2f}'],
                textposition='top center',
            ))
            # SL & TP
            fig.add_hline(y=t['sl'], line_color='rgba(255,23,68,0.6)',
                          line_dash='dot', line_width=1)
            fig.add_hline(y=t['tp'], line_color='rgba(0,230,118,0.6)',
                          line_dash='dot', line_width=1)

    fig.update_layout(
        title=f'Session Manipulation — XAUUSD M5 — {day_date}',
        xaxis_rangeslider_visible=False,
        template='plotly_dark', height=520,
    )
    fig.show()


# Show last 4 trading days
for day in asia_ranges['date'].tail(4):
    plot_session_day(day, df_m5, asia_ranges, trades_df)

TypeError: Addition/subtraction of integers and integer-arrays with Timestamp is no longer supported.  Instead of adding/subtracting `n`, use `n * obj.freq`

### 7.2 — Equity Curve

In [ ]:
def plot_equity(metrics: dict) -> None:
    if not metrics or 'cum_r' not in metrics:
        return
    cum_r = metrics['cum_r']
    dd    = metrics['drawdown']
    closed= metrics['closed'].reset_index(drop=True)

    fig = make_subplots(rows=3, cols=1,
                        row_heights=[0.5, 0.25, 0.25],
                        subplot_titles=['Equity (R)', 'Drawdown', 'Per-Trade PnL'],
                        vertical_spacing=0.08)
    fig.add_trace(go.Scatter(
        x=cum_r.index, y=cum_r.values, mode='lines',
        line=dict(color='#00E5FF', width=2.5),
        fill='tozeroy', fillcolor='rgba(0,229,255,0.08)', name='Equity',
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=cum_r.index, y=cum_r.cummax().values, mode='lines',
        line=dict(color='gold', width=1, dash='dot'), name='Peak',
    ), row=1, col=1)
    fig.add_hline(y=0, line_color='gray', line_dash='dash', row=1, col=1)
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd.values, mode='lines',
        fill='tozeroy', fillcolor='rgba(255,23,68,0.15)',
        line=dict(color='#FF1744', width=1.5), name='DD',
    ), row=2, col=1)
    sess_colors = {'LONDON': '#00E676', 'NY': '#00B0FF'}
    bar_colors  = [sess_colors.get(s, '#888') for s in closed['session']]
    fig.add_trace(go.Bar(
        x=closed.index, y=closed['pnl_r'],
        marker_color=bar_colors, name='PnL',
    ), row=3, col=1)
    fig.add_hline(y=0, line_color='gray', line_dash='dash', row=3, col=1)
    fig.update_layout(
        title=dict(
            text=(f'Session Manipulation — Equity<br>'
                  f'<sup>WR={metrics["win_rate"]*100:.0f}%  '
                  f'Total={metrics["total_r"]:+.1f}R  '
                  f'PF={metrics["profit_factor"]:.2f}</sup>'),
            x=0.5,
        ),
        height=700, template='plotly_dark',
    )
    fig.show()


plot_equity(metrics)

## Final Analysis

In [ ]:
if metrics and metrics.get('total_trades', 0) > 0:
    be_wr = 1 / (1 + RISK_REWARD)
    print('=' * 60)
    print('  SESSION MANIPULATION — FINAL ANALYSIS')
    print('=' * 60)
    print(f'\n[EDGE]')
    print(f'  Break-even WR : {be_wr*100:.1f}%')
    print(f'  Actual WR     : {metrics["win_rate"]*100:.1f}%')
    assessment = '✅ POSITIVE' if metrics['win_rate'] > be_wr else '❌ NEGATIVE'
    print(f'  Edge          : {assessment}')

    print(f'\n[CALENDAR STATS]')
    closed = metrics['closed']
    by_day = closed.groupby(pd.to_datetime(closed['date']).dt.dayofweek).apply(
        lambda x: pd.Series({
            'wr': (x['result']=='TP').mean()*100,
            'n': len(x)
        })
    )
    days = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri'}
    for d, row in by_day.iterrows():
        print(f'  {days.get(d,str(d)):3s}: WR={row["wr"]:.0f}% (n={int(row["n"])})')

    print(f'\n[STRENGTHS]')
    print('  ✓ Based on institutional manipulation patterns')
    print('  ✓ Occurs predictably at session opens')
    print('  ✓ Clear SL placement (beyond sweep extremity)')
    print('  ✓ Works consistently in Gold (high-volume asset)')
    print('  ✓ Only trades during best market hours')

    print(f'\n[WEAKNESSES]')
    print('  ✗ Broker timezone may differ — verify session times')
    print('  ✗ News events can invalidate the pattern')
    print('  ✗ Asia range must be meaningful (min size filter)')
    print('  ✗ Low frequency (1-2 trades per day max)')

    print(f'\n[OPTIMIZATIONS]')
    print('  → Add Asia range size filter (min 5 USD, max 30 USD)')
    print('  → Require sweep candle to be a specific pattern (pin bar)')
    print('  → Use 1-minute TF for more precise entry')
    print('  → Add D1 trend alignment filter')
    print('  → Test different session end times for optimal exit')
    print('=' * 60)